# Evaluate histogram matching

This notebook applies a persisted affine transform, tries several histogram-matching resolutions, and compares intensity quantiles. Histogram matching changes intensities, not geometry.

In [ ]:
# EDIT THESE PATHS before running the notebook.
from pathlib import Path

FIXED_IMAGE = Path("path/to/fixed_reference.tif")
MOVING_IMAGE = Path("path/to/moving_channel.tif")
AFFINE_TRANSFORM = Path("path/to/linear_transform_2.mat")
PIXEL_SIZE_XYZ = (0.15, 0.15, 0.15)
MATCH_POINT_COUNTS = (64, 128, 256, 512)
ITK_THREADS = 1
BLAS_THREADS = 1

In [ ]:
import importlib

from cyexm.io.images import load_ants_image
from cyexm.registration.transforms import apply_affine, import_affine_transform
from cyexm.runtime import NativeThreadConfig, configure_native_threads

NATIVE_THREADS = NativeThreadConfig(ITK_THREADS, BLAS_THREADS)
configure_native_threads(NATIVE_THREADS)
ants = importlib.import_module("ants")
plt = importlib.import_module("matplotlib.pyplot")
np = importlib.import_module("numpy")

fixed = load_ants_image(FIXED_IMAGE, PIXEL_SIZE_XYZ)
moving = load_ants_image(MOVING_IMAGE, PIXEL_SIZE_XYZ)
affine = import_affine_transform(AFFINE_TRANSFORM, native_threads=NATIVE_THREADS)
registered = apply_affine(
    fixed=fixed,
    moving=moving,
    transform=affine,
    interpolation="linear",
    native_threads=NATIVE_THREADS,
)

In [ ]:
quantiles = np.linspace(0.01, 0.99, 99)
fixed_values = fixed.numpy()
fixed_quantiles = np.quantile(fixed_values[fixed_values > 0], quantiles)
results = {}
for point_count in MATCH_POINT_COUNTS:
    matched = ants.histogram_match_image(
        source_image=registered,
        reference_image=fixed,
        number_of_match_points=point_count,
        use_threshold_at_mean_intensity=True,
    )
    values = matched.numpy()
    matched_quantiles = np.quantile(values[values > 0], quantiles)
    results[point_count] = float(np.mean(np.abs(matched_quantiles - fixed_quantiles)))
results

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(fixed.numpy().ravel(), bins=128, alpha=0.5, label="fixed")
axes[0].hist(registered.numpy().ravel(), bins=128, alpha=0.5, label="affine")
axes[0].set_title("Intensity distributions")
axes[0].legend()
axes[1].plot(tuple(results), tuple(results.values()), marker="o")
axes[1].set_xlabel("Histogram match points")
axes[1].set_ylabel("Mean absolute quantile error")
figure.tight_layout()